In [6]:
import os
import json
import random
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
import torch
import torchvision 
import torch.nn as nn
import torch.optim as optim
from torchvision.models import efficientnet_b0
from torchinfo import summary

DATA_DIR = Path('../data/raw/PlantVillage')
print(f"torch version : {torch.__version__}")

torch version : 2.11.0+cpu


In [16]:
import sys
sys.path.append('..')

from src.dataset import get_dataloaders
train_dataloader, val_dataloader, test_dataloader, classes = get_dataloaders(DATA_DIR, 32)

print(f"\nTrain:{len(train_dataloader)}")
print(f"\nVal:{len(val_dataloader)}")
print(f"\nTest:{len(test_dataloader)}")
print(f"\nClasses:{classes}")



Train:516

Val:65

Test:65

Classes:['Pepper__bell___Bacterial_spot', 'Pepper__bell___healthy', 'Potato___Early_blight', 'Potato___Late_blight', 'Potato___healthy', 'Tomato_Bacterial_spot', 'Tomato_Early_blight', 'Tomato_Late_blight', 'Tomato_Leaf_Mold', 'Tomato_Septoria_leaf_spot', 'Tomato_Spider_mites_Two_spotted_spider_mite', 'Tomato__Target_Spot', 'Tomato__Tomato_YellowLeaf__Curl_Virus', 'Tomato__Tomato_mosaic_virus', 'Tomato_healthy']


In [17]:
model = efficientnet_b0(weights='IMAGENET1K_V1')

for param in model.parameters():
    param.requires_grad = False

model.classifier[1] = nn.Linear(in_features=1280, out_features=len(classes))

print(model.classifier)

Downloading: "https://download.pytorch.org/models/efficientnet_b0_rwightman-7f5810bc.pth" to C:\Users\DELL/.cache\torch\hub\checkpoints\efficientnet_b0_rwightman-7f5810bc.pth


100%|███████████████████████████████████████████████████████████████████████████████| 20.5M/20.5M [00:01<00:00, 12.2MB/s]


Sequential(
  (0): Dropout(p=0.2, inplace=True)
  (1): Linear(in_features=1280, out_features=15, bias=True)
)


In [18]:
summary(model, input_size=(1, 3, 224, 224))


Layer (type:depth-idx)                                  Output Shape              Param #
EfficientNet                                            [1, 15]                   --
├─Sequential: 1-1                                       [1, 1280, 7, 7]           --
│    └─Conv2dNormActivation: 2-1                        [1, 32, 112, 112]         --
│    │    └─Conv2d: 3-1                                 [1, 32, 112, 112]         (864)
│    │    └─BatchNorm2d: 3-2                            [1, 32, 112, 112]         (64)
│    │    └─SiLU: 3-3                                   [1, 32, 112, 112]         --
│    └─Sequential: 2-2                                  [1, 16, 112, 112]         --
│    │    └─MBConv: 3-4                                 [1, 16, 112, 112]         (1,448)
│    └─Sequential: 2-3                                  [1, 24, 56, 56]           --
│    │    └─MBConv: 3-5                                 [1, 24, 56, 56]           (6,004)
│    │    └─MBConv: 3-6                      

In [21]:
class_counts = {cls: len(list((DATA_DIR / cls).glob('*.jpg'))) for cls in classes}
total = sum(class_counts.values())
weights = [total / (len(classes) * class_counts[cls]) for cls in classes]
weights_tensor = torch.tensor(weights, dtype=torch.float)
print(weights_tensor)

criterion = nn.CrossEntropyLoss(weight=weights_tensor)
optimizer = optim.Adam(filter(lambda p: p.requires_grad, model.parameters()), lr=0.001)

tensor([1.3799, 0.9314, 1.3757, 1.3757, 9.0509, 0.6468, 1.3757, 0.7210, 1.4451,
        0.7768, 0.8208, 0.9799, 0.4288, 3.6883, 0.8647])


In [22]:
print(criterion)
print(optimizer)

CrossEntropyLoss()
Adam (
Parameter Group 0
    amsgrad: False
    betas: (0.9, 0.999)
    capturable: False
    decoupled_weight_decay: False
    differentiable: False
    eps: 1e-08
    foreach: None
    fused: None
    lr: 0.001
    maximize: False
    weight_decay: 0
)


In [25]:
images, labels = next(iter(train_dataloader))
outputs = model(images)

print(outputs)
print(f"shape: {outputs.shape}")

tensor([[ 1.1041e-01,  9.0729e-02,  3.3264e-01,  1.3929e-02,  1.1676e-01,
          4.3155e-02,  1.7216e-01,  1.2730e-01, -4.1678e-01,  5.2994e-01,
          2.3421e-01, -1.8550e-01, -1.3143e-01, -7.6762e-02,  8.6425e-02],
        [ 3.0822e-01, -2.2322e-01, -2.7782e-01, -2.5531e-02, -4.0254e-01,
          1.2740e-01,  5.9824e-02,  5.0596e-02, -4.9592e-01,  1.1554e-01,
         -1.4274e-01, -2.5042e-01, -2.4882e-01, -2.5222e-01, -1.6773e-01],
        [ 2.1497e-01, -1.2934e-01,  2.9340e-01, -1.1629e-01, -6.1876e-02,
         -2.3737e-01, -1.9929e-01,  8.5855e-02,  2.7917e-01, -1.8365e-01,
          1.1808e-01,  2.1651e-01,  7.2639e-02,  1.2187e-01, -1.7158e-01],
        [-3.8882e-02, -3.1610e-01,  4.2636e-01, -1.9383e-01, -9.2083e-02,
         -6.6047e-01, -3.7108e-02, -1.0956e-02, -1.5523e-01,  2.0676e-01,
         -2.2502e-02, -2.8009e-01,  9.0964e-03,  1.5532e-01,  2.5147e-01],
        [-1.4371e-01, -1.7265e-01,  1.5099e-01,  1.5459e-01,  3.9036e-01,
          3.4391e-01,  1.3117e-01,